# NuWro fake data: is the fit pulled towards the truth?

Companion of `21_prefit_postfit_distributions.ipynb` for the NuWro fake-data fits only. The content of each panel is the same (reconstructed $(\log_{10} Q^2, p_n)$ spectra read from PROfit's `<fit>_v1_PROfile.root`), drawn so that the movement of the prediction towards the fake-data truth is the message:

* **Pre-fit** (blue, dashed, hatched band): the GENIE central-value prediction (`*_cv*`) with the pre-fit error band (`*_preerrband*`).
* **Post-fit** (red, solid, filled band): PROfit's constrained best fit (`*_bestfit*`) with the post-fit band (`*_posterrband*`).
* **NuWro fake data** (black points): the fitted sample (`*_data*`). It is the unfluctuated NuWro prediction scaled to the XML POT, so it *is* the truth the fit should recover; the error bars are $\sqrt{N}$ of the fake data.

The lower panels are ratios **to the NuWro fake data**, not to the pre-fit prediction as in notebook 21: the truth sits at 1 and a fit that pulls in the right direction moves the red line from the blue one towards 1.

For each fit two figures are written to `figs/nuwro_fit_results/<fit>/`: the projections (`nuwro_truth_projections.pdf`) and the $\log_{10} Q^2$ spectra in slices of $p_n$ (`nuwro_truth_slices.pdf`). The $\chi^2$ annotation is the post-fit one of notebook 21 (PROfit's $\chi^2_\mathrm{min}$ including pulls, split into stat + syst + pull, over $\nu = N_\mathrm{bins} - N_\mathrm{flat}$).

The table at the end quantifies the movement with the statistics-only distance to the truth over the 72 fitted bins, $\sum_i (d_i - \mu_i)^2 / d_i$, for the pre-fit CV and for the drawn post-fit prediction (the latter is the stat term of the $\chi^2$ split). It is a distance, not a goodness of fit: it ignores the systematic covariance. The same pre-fit $\to$ post-fit pair is printed on every figure.


In [ ]:
from pathlib import Path
import re
import os
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

SCRIPTS = next(p / "python" / "scripts" for p in [Path.cwd(), *Path.cwd().parents] if (p / "python" / "scripts").is_dir())
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))
from chi2_split import TOLERANCE, fit_metric, stat_syst_split, stat_chi2, warn_if_mismatch

mpl.rcParams.update({
    "figure.dpi": 140,
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
    "mathtext.fontset": "stix",
})

In [ ]:
# Inputs. The constants mirror 5_publication_plots.ipynb and 3_fit_chi2_summary.ipynb so the
# header, the POT and the chi^2 bookkeeping agree with the other figures.
FIT_RESULTS_ROOT = Path(os.environ.get("AXIAL_MASS_OUTPUT_ROOT", "/nevis/hopper/data/epelaez/axial_mass_cnpmu_iterfrozen"))
CHANNEL = "nu_uBooNE_numuCC1p"

# NuWro fake data, normalized to 1.75e21 POT in the XML.
CFG = dict(suite="nuwro", data_label="NuWro fake data (truth)", target_pot=1.75e21, header="Simulation")

def pot_text(pot):
    """POT for figure labels in mathtext, e.g. $9.32 \cdot 10^{19}$ POT."""
    exp = int(np.floor(np.log10(pot)))
    return rf"${pot / 10**exp:.2f} \cdot 10^{{{exp}}}$ POT"

# Fits to draw, in display order, and their legend names (FIT_ORDER of 3_fit_chi2_summary.ipynb).
FITS = [
    "ma", "ma_uniform", "minerva_k8", "minerva_k7",
    "minerva_k6", "lqcd_k7", "lqcd_k6", "minerva_lqcd_k7", "minerva_lqcd_k6",
    "minerva_k6_nuisance", "lqcd_k6_nuisance", "minerva_lqcd_k6_nuisance",
    "minerva_k6_uniform", "minerva_k6_uniform_nuisance",
]
FIT_LABELS = {
    "ma": r"$M_A$ Gaussian prior",
    "ma_uniform": r"$M_A$ uniform prior",
    "minerva_k8": r"$\bar{\nu}$–H prior, $k_{\max}=8$",
    "minerva_k7": r"$\bar{\nu}$–H prior, $k_{\max}=7$",
    "minerva_k6": r"$\bar{\nu}$–H prior, $k_{\max}=6$",
    "lqcd_k6": r"LQCD prior, $k_{\max}=6$",
    "lqcd_k7": r"LQCD prior, $k_{\max}=7$",
    "minerva_lqcd_k7": r"$\bar{\nu}$–H + LQCD prior, $k_{\max}=7$",
    "minerva_lqcd_k6": r"$\bar{\nu}$–H + LQCD prior, $k_{\max}=6$",
    "minerva_k6_nuisance": r"$\bar{\nu}$–H prior w/ nuisance, $k_{\max}=6$",
    "lqcd_k6_nuisance": r"LQCD prior w/ nuisance, $k_{\max}=6$",
    "minerva_lqcd_k6_nuisance": r"$\bar{\nu}$–H + LQCD prior w/ nuisance, $k_{\max}=6$",
    "minerva_k6_uniform": r"Uniform prior, $k_{\max}=6$",
    "minerva_k6_uniform_nuisance": r"Uniform prior w/ nuisance, $k_{\max}=6$",
}
# Parameters with a flat prior contribute no pull term to PROfit's best-fit chi^2.
UNIFORM_PARAMETERS = {
    "ma_uniform": {"MACCQE"},
    "minerva_k6_uniform": {"FAzexpMinervaK6PCA1", "FAzexpMinervaK6PCA2"},
    "minerva_k6_uniform_nuisance": {"FAzexpMinervaK6PCA1", "FAzexpMinervaK6PCA2"},
}
CHI2_PATTERN = re.compile(r"Global Best Fit chi(?:\^?2|²):\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)")

search_root = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (search_root, *search_root.parents)
     if (path / "python" / "notebooks").is_dir() and (path / "xml").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the ma_zexp repository root")
FIGURE_ROOT = REPO_ROOT / "figs"


def fit_directory(cfg, fit):
    return FIT_RESULTS_ROOT / f"{cfg['suite']}_fit_results" / fit

In [ ]:
# Layout shared with 21_prefit_postfit_distributions.ipynb; the colours differ: pre-fit blue
# (dashed line, hatched band), post-fit red (solid line, filled band), NuWro truth black points.
BACKGROUND = "#FFFFFF"
FOREGROUND = "#303642"
ACCENT = "#020202"
PREFIT_COLOR = "#2a78d6"
PREFIT_FACECOLOR = mpl.colors.to_rgba(PREFIT_COLOR, 0.15)
PREFIT_HATCHCOLOR = mpl.colors.to_rgba(PREFIT_COLOR, 0.60)
PREFIT_HATCH = "/////"
POSTFIT_COLOR = "#d1343a"
POSTFIT_FACECOLOR = mpl.colors.to_rgba(POSTFIT_COLOR, 0.35)
FIGURE_RC = {
    "font.family": "sans-serif", "mathtext.fontset": "dejavusans", "hatch.linewidth": 0.5,
}
Q2_LABEL = r"$\log_{10}\!\left(Q^2_{\mathrm{reco}}/\mathrm{GeV}^2\right)$"
PN_LABEL = r"$p_n^{\mathrm{reco}}$ [GeV/$c$]"
# Outer limits of the ratio panels (the window adapts to the pre-fit and post-fit lines inside them).
RATIO_LIMITS = (0.0, 2.5)


def _edge_decimals(edges, minimum=1, maximum=4):
    for decimals in range(minimum, maximum + 1):
        if np.allclose(np.round(edges, decimals), edges, rtol=0, atol=1e-9):
            return decimals
    return maximum


def _style_axis(ax, labelsize=None):
    ax.set_facecolor(BACKGROUND)
    ax.tick_params(which="both", direction="in", top=True, right=True,
                   colors=FOREGROUND, labelcolor=FOREGROUND, length=6)
    if labelsize is not None:
        ax.tick_params(labelsize=labelsize)
    for spine in ax.spines.values():
        spine.set_color(FOREGROUND)
        spine.set_linewidth(1.0)


def _band(ax, edges, lower, upper, *, facecolor, edgecolor, hatch, zorder):
    lower = np.clip(lower, 0, None) if hatch else lower
    ax.fill_between(
        edges, np.r_[lower, lower[-1]], np.r_[upper, upper[-1]], step="post",
        facecolor=facecolor, edgecolor=edgecolor, hatch=hatch, linewidth=0, zorder=zorder,
    )

In [ ]:
def _graph_band(graph):
    """(centre, lower, upper) of a PROfit TGraphAsymmErrors band."""
    centre = graph.values(axis="y")
    return centre, centre - graph.errors("low", axis="y"), centre + graph.errors("high", axis="y")


def _read_spectrum(profile_file, suffix):
    """CV, best fit, data and both bands of one PROfit histogram family (``suffix`` is '' for
    the first-variable projection, '_y' for the second, '_slice_ybinN' for a slice)."""
    cv = profile_file[f"{CHANNEL}_cv1d{suffix}" if suffix in ("", "_y") else f"{CHANNEL}_cv{suffix}"]
    bestfit = profile_file[f"{CHANNEL}_bestfit{suffix}"]
    data = profile_file[f"{CHANNEL}_data{suffix}"]
    _, pre_lower, pre_upper = _graph_band(profile_file[f"{CHANNEL}_preerrband{suffix}"])
    post_centre, post_lower, post_upper = _graph_band(profile_file[f"{CHANNEL}_posterrband{suffix}"])
    # Current PROfit anchors the post-fit band on the drawn best fit (data-constrained); older
    # outputs did the same without the constraint. Either way the band graph is the reference.
    if not np.allclose(post_centre, bestfit.values(), rtol=1e-3, atol=1e-3):
        print(f"  note: post-fit band centre differs from the best-fit histogram for {suffix or 'x'}; "
              "drawing the band centre as the post-fit prediction")
    return dict(
        edges=np.round(cv.axis().edges(), 6), prefit=cv.values(), postfit=post_centre,
        data=data.values(), data_error=data.errors(),
        prefit_lower=pre_lower, prefit_upper=pre_upper,
        postfit_lower=post_lower, postfit_upper=post_upper,
    )


def _postfit_split(profile_path, chi2_post):
    """Split PROfit's chi2_min into (stat, syst, param), see python/scripts/chi2_split.py. stat:
    residual of the drawn constrained best fit (bestfit2d) with the statistical variance of the
    fit's statistic; syst: squared posterior pulls of the covariance-type systematic modes
    (covmode_pulls); param: Gaussian pulls of the fitted parameters. Neyman: stat + syst equals
    PROfit's total minus param exactly (Woodbury) and the mismatch is checked. CNP: stat is the
    total minus param minus syst, with a coarse closure guard."""
    if chi2_post is None:
        return chi2_post
    split = stat_syst_split(profile_path, chi2_post["data"])
    if not np.isfinite(split["syst"]):
        return chi2_post
    warn_if_mismatch(split)
    # Distance to the truth: always sum (d - mu)^2 / d of the drawn best fit, whatever the fit's
    # statistic (equal to stat for a Neyman fit).
    with uproot.open(profile_path) as profile_file:
        data = profile_file[f"{CHANNEL}_data2d"].values()
        bestfit = profile_file[f"{CHANNEL}_bestfit2d"].values()
    distance = stat_chi2(data, bestfit, "neyman")
    return dict(chi2_post, stat=split["stat"], syst=split["syst"], distance=distance)


def _profit_chi2(log_path, fit, parameters):
    """PROfit's global minimum chi^2 split as (total, data term, pull term). The pull term is the
    Gaussian penalty of the fitted parameters (flat-prior ones carry none), as in
    3_fit_chi2_summary.ipynb. The total is the goodness of fit of the drawn constrained best fit."""
    if not log_path.is_file():
        return None
    matches = CHI2_PATTERN.findall(log_path.read_text(errors="replace"))
    if not matches:
        return None
    uniform = UNIFORM_PARAMETERS.get(fit, set())
    pull = sum(value ** 2 for name, value in parameters.items() if name not in uniform)
    total = float(matches[-1])
    return dict(total=total, data=total - pull, pull=pull)


def load_fit_spectra(cfg, fit):
    """Everything the two figures need for one (family, fit); None when the fit has not run."""
    directory = fit_directory(cfg, fit)
    profile_path = directory / f"{fit}_v1_PROfile.root"
    if not profile_path.is_file():
        print(f"{cfg['suite']}/{fit}: {profile_path.name} missing, skipped")
        return None
    with uproot.open(profile_path) as profile_file:
        cv_2d = profile_file[f"{CHANNEL}_cv2d"]
        # PROfit writes single-precision axes; round so 0.1 does not print as 0.1000000015.
        q2_edges, pn_edges = (np.round(axis.edges(), 6) for axis in cv_2d.axes)
        data_2d = profile_file[f"{CHANNEL}_data2d"].values()
        cv_values = cv_2d.values()
        projections = {"q2": _read_spectrum(profile_file, ""), "pn": _read_spectrum(profile_file, "_y")}
        n_pn_bins = len(pn_edges) - 1
        if f"{CHANNEL}_bestfit_slice_ybin1" in profile_file:
            slices = [_read_spectrum(profile_file, f"_slice_ybin{pn_bin + 1}") for pn_bin in range(n_pn_bins)]
        else:
            print(f"{cfg['suite']}/{fit}: no slice histograms in {profile_path.name}; slices figure skipped")
            slices = None
        global_fit = profile_file["global_fit_result"]
        parameters = dict(zip(global_fit.axis().labels(), global_fit.values()))
        one_sigma = profile_file["one_sigma_errs"]
        errors_low = one_sigma.errors("low", axis="y")
        errors_high = one_sigma.errors("high", axis="y")

    n_bins = data_2d.size
    # nu as in 3_fit_chi2_summary.ipynb: only flat-prior parameters cost a degree of freedom.
    ndof = n_bins - len(UNIFORM_PARAMETERS.get(fit, ()))
    # Statistics-only distance of the pre-fit CV to the NuWro truth, same bins and weights as the
    # stat term of the post-fit split.
    use = data_2d > 0
    stat_pre = float(np.sum((data_2d[use] - cv_values[use]) ** 2 / data_2d[use]))
    chi2_post = _profit_chi2(directory / "profile.log", fit, parameters)
    chi2_post = _postfit_split(profile_path, chi2_post)
    return dict(
        cfg=cfg, fit=fit, label=FIT_LABELS.get(fit, fit), q2_edges=q2_edges, pn_edges=pn_edges,
        projections=projections, slices=slices, n_bins=n_bins, ndof=ndof,
        stat_pre=stat_pre, chi2_post=chi2_post, parameters=parameters,
        parameter_errors=dict(zip(parameters, zip(errors_low, errors_high))),
    )

In [ ]:
def _draw_spectrum(ax, spec):
    """Pre-fit (blue hatched band, dashed CV), post-fit (red filled band, solid line), NuWro truth."""
    edges = spec["edges"]
    centres = 0.5 * (edges[:-1] + edges[1:])
    _band(ax, edges, spec["prefit_lower"], spec["prefit_upper"], facecolor=PREFIT_FACECOLOR,
          edgecolor=PREFIT_HATCHCOLOR, hatch=PREFIT_HATCH, zorder=4)
    ax.stairs(spec["prefit"], edges, baseline=None, color=PREFIT_COLOR, linestyle="--",
              linewidth=1.2, zorder=5)
    _band(ax, edges, spec["postfit_lower"], spec["postfit_upper"], facecolor=POSTFIT_FACECOLOR,
          edgecolor="none", hatch=None, zorder=6)
    ax.stairs(spec["postfit"], edges, baseline=None, color=POSTFIT_COLOR, linestyle="-",
              linewidth=1.6, zorder=7)
    ax.errorbar(
        centres, spec["data"], xerr=0.5 * np.diff(edges), yerr=spec["data_error"],
        fmt="o", color="black", markerfacecolor="black", markersize=3.8, elinewidth=1.0,
        capsize=2.0, linestyle="none", zorder=10,
    )
    ax.set_xlim(edges[[0, -1]])
    upper = max(spec["prefit_upper"].max(), spec["postfit_upper"].max(),
                (spec["data"] + spec["data_error"]).max())
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5, integer=True))
    return upper


def _draw_ratio(ax, spec):
    """The same three ingredients divided by the NuWro fake data, so the truth sits at 1."""
    edges = spec["edges"]
    centres = 0.5 * (edges[:-1] + edges[1:])
    reference = spec["data"]
    safe = np.where(reference > 0, reference, np.nan)
    ratio = lambda values: values / safe
    _band(ax, edges, ratio(spec["prefit_lower"]), ratio(spec["prefit_upper"]),
          facecolor=PREFIT_FACECOLOR, edgecolor=PREFIT_HATCHCOLOR, hatch=PREFIT_HATCH, zorder=4)
    ax.stairs(ratio(spec["prefit"]), edges, baseline=None, color=PREFIT_COLOR, linestyle="--",
              linewidth=1.2, zorder=5)
    _band(ax, edges, ratio(spec["postfit_lower"]), ratio(spec["postfit_upper"]),
          facecolor=POSTFIT_FACECOLOR, edgecolor="none", hatch=None, zorder=6)
    ax.stairs(ratio(spec["postfit"]), edges, baseline=None, color=POSTFIT_COLOR, linewidth=1.6, zorder=7)
    ax.axhline(1.0, color="black", linewidth=0.8, zorder=8)
    ax.errorbar(
        centres, ratio(spec["data"]), xerr=0.5 * np.diff(edges), yerr=ratio(spec["data_error"]),
        fmt="o", color="black", markerfacecolor="black", markersize=3.8, elinewidth=1.0,
        capsize=2.0, linestyle="none", zorder=10,
    )
    # Window from the two predictions and the truth's error bars (not the bands), symmetric about
    # 1 where possible, so the pre-fit line stays on scale even where it is far from NuWro.
    extremes = np.concatenate([
        ratio(spec["data"] - spec["data_error"]), ratio(spec["data"] + spec["data_error"]),
        ratio(spec["prefit"]), ratio(spec["postfit"]),
    ])
    extremes = extremes[np.isfinite(extremes)]
    half_width = 1.15 * max(abs(extremes - 1.0).max() if extremes.size else 0.2, 0.1)
    half_width = min(half_width, RATIO_LIMITS[1] - 1.0)
    ax.set_ylim(max(1.0 - half_width, RATIO_LIMITS[0]), 1.0 + half_width)
    ax.set_xlim(edges[[0, -1]])
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=4))
    ax.grid(axis="y", color="#9AA4B2", alpha=0.22, linewidth=0.7)


def _legend_entries(data_label):
    handles = [
        (mpl.patches.Patch(facecolor=PREFIT_FACECOLOR, edgecolor=PREFIT_COLOR, hatch=PREFIT_HATCH,
                           linestyle="--", linewidth=1.2)),
        (mpl.patches.Patch(facecolor=POSTFIT_FACECOLOR, edgecolor=POSTFIT_COLOR, linewidth=1.6)),
        mpl.lines.Line2D([], [], color="black", marker="o", linestyle="none", markersize=4.0),
    ]
    labels = ["Pre-fit prediction, total unc.", "Post-fit prediction, post-fit unc.", data_label]
    return handles, labels


def _chi2_text(st):
    parts = []
    ndof = st["ndof"]
    # Post-fit chi^2 as in notebook 21.
    # Two decimals and no pull term for fits without a Gaussian-constrained parameter, as in the
    # LaTeX table of 3_fit_chi2_summary.ipynb.
    post = st["chi2_post"]
    no_gaussian = set(st["parameters"]) <= UNIFORM_PARAMETERS.get(st["fit"], set())
    if post is not None and "stat" in post:
        pull = "" if no_gaussian else rf" + {post['pull']:.2f}_{{\,\mathrm{{pull}}}}"
        parts.append(rf"$\chi^2_{{\mathrm{{post}}}}/\nu = ({post['stat']:.2f}_{{\,\mathrm{{stat}}}} + "
                     rf"{post['syst']:.2f}_{{\,\mathrm{{syst}}}}{pull})"
                     rf"/{ndof} = {post['total']:.2f}/{ndof}$")
    elif post is not None:
        parts.append(rf"$\chi^2_{{\mathrm{{post}}}}/\nu = {post['total']:.2f}/{ndof}$")
    return "\n".join(parts)


def _distance_text(st):
    """Statistics-only distance to the NuWro truth over all fitted bins, pre-fit -> post-fit."""
    post = st["chi2_post"] or {}
    if "distance" not in post:
        return ""
    return (rf"$\sum_i (d_i-\mu_i)^2/d_i$ to NuWro: ${st['stat_pre']:.1f}$ pre-fit "
            rf"$\to$ ${post['distance']:.1f}$ post-fit")


def _fit_columns(fig, outer, axes, n_cols, pad=0.10):
    """Widen the gaps between the slice columns until the y tick labels of every panel right of
    the first column clear the neighbouring panel (``pad`` inches spare). The label width depends
    on the event counts, hence on the POT of the fitted sample."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    needed = 0.0
    for ax in axes:
        if ax.get_subplotspec().get_topmost_subplotspec().colspan.start == 0:
            continue
        labels_left = ax.yaxis.get_tightbbox(renderer).x0
        needed = max(needed, (ax.get_window_extent(renderer).x0 - labels_left) / fig.dpi)
    gap = needed + pad
    params = outer.get_subplot_params(fig)
    available = (params.right - params.left) * fig.get_figwidth()
    # n panels of width w and n - 1 gaps of wspace * w fill the available width.
    wspace = gap * n_cols / (available - (n_cols - 1) * gap)
    if wspace > params.wspace:
        outer.update(wspace=wspace)


def _header(fig, st, header_y, left_x):
    cfg = st["cfg"]
    fig.text(0.5, header_y, st["label"], color=ACCENT, fontsize=14, ha="center", va="bottom")
    fig.text(0.975, header_y, pot_text(cfg['target_pot']), color=ACCENT,
             fontsize=14, ha="right", va="bottom")


def _save(fig, st, name):
    directory = FIGURE_ROOT / f"{st['cfg']['suite']}_fit_results" / st["fit"]
    directory.mkdir(parents=True, exist_ok=True)
    fig.savefig(directory / name, format="pdf", dpi=600, bbox_inches="tight", facecolor="white")

In [ ]:
def plot_projections(st):
    """log10 Q^2 and p_n projections with ratio panels; writes nuwro_truth_projections.pdf."""
    panels = [(st["projections"]["q2"], Q2_LABEL), (st["projections"]["pn"], PN_LABEL)]
    with mpl.rc_context(FIGURE_RC):
        fig = plt.figure(figsize=(13.0, 6.4), facecolor=BACKGROUND)
        grid = fig.add_gridspec(2, 2, height_ratios=[3.0, 1.15], hspace=0.06, wspace=0.16,
                                left=0.07, right=0.975, bottom=0.11, top=0.90)
        for column, (spec, xlabel) in enumerate(panels):
            ax = fig.add_subplot(grid[0, column])
            ax_ratio = fig.add_subplot(grid[1, column], sharex=ax)
            upper = _draw_spectrum(ax, spec)
            ax.set_ylim(0, 1.45 * upper)
            _draw_ratio(ax_ratio, spec)
            ax.tick_params(labelbottom=False)
            _style_axis(ax)
            _style_axis(ax_ratio)
            ax_ratio.set_xlabel(xlabel, color=FOREGROUND, fontsize=16, labelpad=5)
            ax.set_ylabel("Events / bin", color=FOREGROUND, fontsize=16)
            ax_ratio.set_ylabel("Ratio to NuWro", color=FOREGROUND, fontsize=12)
            if column == 0:
                handles, labels = _legend_entries(st["cfg"]["data_label"])
                legend = ax.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.025, 0.975),
                                   frameon=False, fontsize=10, labelspacing=0.4, handlelength=1.6,
                                   borderaxespad=0)
                for text in legend.get_texts():
                    text.set_color(FOREGROUND)
            chi2_text = "\n".join(filter(None, [_chi2_text(st), _distance_text(st)])) if column == 1 else ""
            if chi2_text:
                ax.text(0.975, 0.965, chi2_text, transform=ax.transAxes, ha="right", va="top",
                        fontsize=11, color=FOREGROUND, linespacing=1.4)
        _header(fig, st, header_y=0.915, left_x=0.07)
        _save(fig, st, "nuwro_truth_projections.pdf")
        plt.show()


def plot_slices(st):
    """log10 Q^2 spectra in p_n slices with ratio panels; writes nuwro_truth_slices.pdf."""
    if st["slices"] is None:
        return
    slices, pn_edges = st["slices"], st["pn_edges"]
    n_pn_bins = len(slices)
    n_cols = 4
    n_rows = -(-n_pn_bins // n_cols)
    pn_decimals = _edge_decimals(pn_edges)

    with mpl.rc_context(FIGURE_RC):
        fig = plt.figure(figsize=(13.0, 5.0 * n_rows), facecolor=BACKGROUND)
        outer = fig.add_gridspec(n_rows, n_cols, hspace=0.16, wspace=0.10,
                                 left=0.06, right=0.975, bottom=0.07, top=0.865)
        main_axes, ratio_axes = [], []
        for pn_bin, spec in enumerate(slices):
            row, column = divmod(pn_bin, n_cols)
            inner = outer[row, column].subgridspec(2, 1, height_ratios=[3.0, 1.15], hspace=0.05)
            ax = fig.add_subplot(inner[0])
            ax_ratio = fig.add_subplot(inner[1], sharex=ax)
            upper = _draw_spectrum(ax, spec)
            ax.set_ylim(0, 1.35 * upper)
            _draw_ratio(ax_ratio, spec)
            ax.tick_params(labelbottom=False)
            _style_axis(ax, labelsize=10)
            _style_axis(ax_ratio, labelsize=10)
            ax.text(
                0.95, 0.95,
                rf"${pn_edges[pn_bin]:.{pn_decimals}f} \leq p_n^\mathrm{{reco}} "
                rf"< {pn_edges[pn_bin + 1]:.{pn_decimals}f}$ GeV/$c$",
                transform=ax.transAxes, ha="right", va="top", fontsize=10, color=FOREGROUND,
            )
            if column == 0:
                ax.set_ylabel("Events / bin", color=FOREGROUND, fontsize=13)
                ax_ratio.set_ylabel("Ratio to NuWro", color=FOREGROUND, fontsize=11)
            if row == n_rows - 1 or pn_bin + n_cols >= n_pn_bins:
                ax_ratio.set_xlabel(Q2_LABEL, color=FOREGROUND, fontsize=13, labelpad=4)
            main_axes.append(ax)
            ratio_axes.append(ax_ratio)

        handles, labels = _legend_entries(st["cfg"]["data_label"])
        legend = fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.936),
                            ncol=len(labels), frameon=False, fontsize=11, handlelength=1.6,
                            columnspacing=1.8, borderaxespad=0)
        for text in legend.get_texts():
            text.set_color(FOREGROUND)
        # chi^2 and distance to the truth on one line below the legend.
        fig.text(0.5, 0.912, ",    ".join(filter(None, [_chi2_text(st), _distance_text(st)])),
                 ha="center", va="top", fontsize=11, color=FOREGROUND)
        _fit_columns(fig, outer, main_axes + ratio_axes, n_cols)
        _header(fig, st, header_y=0.953, left_x=0.06)
        _save(fig, st, "nuwro_truth_slices.pdf")
        plt.show()

## All NuWro fits

Edit `FITS` above to change what is drawn. Fits whose `PROfile.root` does not exist yet are reported and skipped. In the table, `stat_pre` and `stat_post` are the statistics-only distances to the NuWro truth before and after the fit; the remaining columns are the post-fit $\chi^2$ split of notebook 21.


In [ ]:
records = []
for fit in FITS:
    st = load_fit_spectra(CFG, fit)
    if st is None:
        continue
    post = st["chi2_post"] or {}
    nan = float("nan")
    print(f"nuwro/{fit}: {st['n_bins']} bins, stat-only distance to NuWro {st['stat_pre']:.1f} (pre-fit) -> "
          f"{post.get('distance', nan):.1f} (post-fit)")
    plot_projections(st)
    plot_slices(st)
    records.append(dict(fit=fit, stat_pre=st["stat_pre"], stat_post=post.get("distance"),
                        chi2_post_syst=post.get("syst"), chi2_post_pull=post.get("pull"),
                        chi2_post=post.get("total"), n_bins=st["n_bins"], ndof=st["ndof"]))

summary = pd.DataFrame(records).set_index("fit")
display(summary.style.format({"stat_pre": "{:.1f}", "stat_post": "{:.1f}", "chi2_post_syst": "{:.1f}",
                              "chi2_post_pull": "{:.2f}", "chi2_post": "{:.1f}"}, na_rep="n/a"))
